# Toy text-classification datasets — exploration

Quick stats and sample viewer for **20 Newsgroups** and **RCV1**, plus pointers to other small/medium datasets worth knowing.

## Other toy-ish datasets to know about

| Dataset | Size | Labels | Multi-label? | Notes |
|---|---|---|---|---|
| **AG News** | 120k train / 7.6k test | 4 | no | Short news titles+descriptions. Standard quick benchmark. (`datasets` lib: `ag_news`) |
| **DBpedia-14** | 560k / 70k | 14 | no | Wikipedia article ontology classes. Cleaner than 20NG. |
| **BBC News** | ~2.2k | 5 | no | Tiny, very clean — great smoke-test dataset. |
| **IMDB** | 50k | 2 | no | Sentiment, longer-form than AG. |
| **Yelp / Amazon Polarity** | 100k–4M | 2 or 5 | no | Scales nicely if you outgrow toys. |
| **OHSUMED** | ~50k | 23 (MeSH) | yes | Medical abstracts; closer in feel to RCV1. |
| **arXiv abstracts** (Kaggle / HF `ccdv/arxiv-classification`) | varies | dozens | yes (categories) | **Relevant to paper2data** — actual scientific papers. |
| **PubMed / MEDLINE abstracts** | huge | MeSH terms | yes | Highly multi-label, biomedical. |
| **Reuters-21578** | ~10k | 90 | yes | Older, smaller sibling of RCV1. Loadable via NLTK. |

Given the project name `paper2data`, **arXiv abstracts** is probably the most ecologically valid toy — papers in, structured categories out — once you outgrow 20NG/RCV1.

In [7]:
import numpy as np
import random
from sklearn.datasets import fetch_20newsgroups, fetch_rcv1

random.seed(0)

## 1. 20 Newsgroups

In [8]:
ng = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes'))
lens = np.array([len(t.split()) for t in ng.data])

print(f'documents       : {len(ng.data):,}')
print(f'unique labels   : {len(ng.target_names)}')
print(f'labels per doc  : 1 (single-label, mutually exclusive)')
print(f'words / doc     : mean={lens.mean():.1f}  median={np.median(lens):.0f}  p90={np.percentile(lens,90):.0f}  max={lens.max()}')
print()
print('categories:')
for i, name in enumerate(ng.target_names):
    print(f'  {i:2d}  {name}')

documents       : 18,846
unique labels   : 20
labels per doc  : 1 (single-label, mutually exclusive)
words / doc     : mean=181.6  median=83  p90=331  max=11765

categories:
   0  alt.atheism
   1  comp.graphics
   2  comp.os.ms-windows.misc
   3  comp.sys.ibm.pc.hardware
   4  comp.sys.mac.hardware
   5  comp.windows.x
   6  misc.forsale
   7  rec.autos
   8  rec.motorcycles
   9  rec.sport.baseball
  10  rec.sport.hockey
  11  sci.crypt
  12  sci.electronics
  13  sci.med
  14  sci.space
  15  soc.religion.christian
  16  talk.politics.guns
  17  talk.politics.mideast
  18  talk.politics.misc
  19  talk.religion.misc


In [9]:
def show_20ng_examples(n=3, category=None):
    """Print n random 20NG docs. If category given (name or int), filter to that class."""
    if category is None:
        idxs = random.sample(range(len(ng.data)), n)
    else:
        cat_idx = ng.target_names.index(category) if isinstance(category, str) else category
        pool = np.where(ng.target == cat_idx)[0]
        idxs = random.sample(list(pool), min(n, len(pool)))
    for i in idxs:
        print('=' * 80)
        print(f'idx={i}  label={ng.target_names[ng.target[i]]}  ({len(ng.data[i].split())} words)')
        print('-' * 80)
        print(ng.data[i].strip())
        print()

show_20ng_examples(n=25)

idx=12623  label=talk.religion.misc  (50 words)
--------------------------------------------------------------------------------
And he went out to meet Asa,
	And said unto him,
	Hear ye me, Asa,
	And all Judah and Benjamin;
	The LORD is with you, while ye be with him;
	and if ye seek him, he will be found of you;
	but if ye forsake him, he will forsake you.

idx=13781  label=sci.med  (130 words)
--------------------------------------------------------------------------------
As a matter of fact, I just saw a dermatologist the other day, and while I 
was there, I asked him about dry skin. I'd been spending a small fortune
on various creams, lotions, and other dry skin treatments.
He said all I needed was a large jar of vaseline. Soak in a lukewarm tub
of water for 10 minutes (ONLY 10 minutes!) then massage in the vaseline,
to trap the moisture in. That will help. I haven't tried it yet, but you
can bet I will. The hard part will be finding the time to rub in the
vaseline properly. If i

In [10]:
# Browse a specific category — change the string to anything from the list above
show_20ng_examples(n=2, category='sci.space')

idx=2029  label=sci.space  (639 words)
--------------------------------------------------------------------------------
(sci.space readers can skip the first paragraph)

  Yesterday, in response to Henry Spencer's question about the 
temperature of a blackbody in interstellar space, I said "Dust grains
acts as blackbodies, and they're at 40-150 K."  Well, I was dead
wrong.  Our local interstellar dust expert, Bruce Draine, has
informed me that dust grains _aren't_ good radiators in the far IR,
which is why they are so warm; actually, the ambient radiation field
from distant stars can bring a true blackbody to only 3 or 4 Kelvin.
Sorry, Henry, and anyone else I misled.  Obviously, time for me to
take another ISM class :-(

  In other news, Alan Stern of the Southwest Research Institute gave
a talk on the Pluto-Charon binary system yesterday.  He gave a brief
overview of the currently-accepted system parameters (volume ratio of
about 8:1, mass ratio about 15:1 or so, plus lots more...) a

### Readable Reuters samples via NLTK (Reuters-21578)

Smaller (~10k docs, 90 topics, multi-label) but you get the original text.

In [11]:
import nltk
nltk.download('reuters', quiet=True)
from nltk.corpus import reuters

fileids = reuters.fileids()
r_lens = np.array([len(reuters.raw(f).split()) for f in fileids])
r_ncats = np.array([len(reuters.categories(f)) for f in fileids])

print(f'documents       : {len(fileids):,}')
print(f'unique labels   : {len(reuters.categories())}')
print(f'labels per doc  : mean={r_ncats.mean():.2f}  median={np.median(r_ncats):.0f}  max={r_ncats.max()}')
print(f'words / doc     : mean={r_lens.mean():.1f}  median={np.median(r_lens):.0f}  p90={np.percentile(r_lens,90):.0f}  max={r_lens.max()}')

print()
for fid in random.sample(fileids, 3):
    print('=' * 80)
    print(f'{fid}  categories={reuters.categories(fid)}  ({len(reuters.raw(fid).split())} words)')
    print('-' * 80)
    print(reuters.raw(fid).strip())
    print()

documents       : 10,788
unique labels   : 90
labels per doc  : mean=1.24  median=1  max=15
words / doc     : mean=127.8  median=84  p90=289  max=1672

training/4267  categories=['coffee']  (576 words)
--------------------------------------------------------------------------------
NO NEAR TERM BRAZIL COFFEE MOVES EXPECTED
  The Brazilian Coffee Institute,
  IBC, is unlikely to disclose its future export policy until the
  end of next week at the earliest, trade sources said.
      IBC president Jorio Dauster is meeting government
  ministers, producers, exporters and market analysts to assess
  Brazil's position in the light of the failure of talks in
  London earlier this month to set new International Coffee
  Organization, ICO, export quotas.
      "The failure of the talks means Brazil has got to rethink
  its position completely," one Santos exporter said.
      A meeting of the National Coffee Policy Council is set for
  Thursday, March 19, and Dauster will almost certainly expl